## Predict Top 5 UP Stocks
Loads `stock_model_5d.joblib` and `stock_model_14d.joblib`, fetches live data for every ticker in a CSV you supply, and ranks them by **P(UP > 3%)** confidence.

In [1]:
import sys
import warnings
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

warnings.filterwarnings('ignore')

# Make sure project modules are importable
project_dir = Path('.').resolve()
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from get_price_data import fetch_latest_data, calculate_technical_indicators
from train_model import FEATURE_NAMES, RANK_FEATURE_NAMES, ALL_FEATURE_NAMES, _ticker_exchange

print('Imports OK')

ModuleNotFoundError: No module named 'yfinance'

In [ ]:
MODEL_5D_PATH  = project_dir / 'stock_model_5d.joblib'
MODEL_14D_PATH = project_dir / 'stock_model_14d.joblib'

def _load(path):
    if not Path(path).exists():
        raise FileNotFoundError(f'Model file not found: {path}')
    return joblib.load(path)

model_5d  = _load(MODEL_5D_PATH)
model_14d = _load(MODEL_14D_PATH)

print(f'5-day  model : {model_5d["description"]}')
print(f'14-day model : {model_14d["description"]}')

In [ ]:
csv_path = input('Enter path to ticker CSV file: ').strip().strip('"').strip("'")

csv_path = Path(csv_path)
if not csv_path.exists():
    raise FileNotFoundError(f'File not found: {csv_path}')

tickers = pd.read_csv(csv_path, header=None).iloc[:, 0].dropna().str.strip().tolist()
print(f'Loaded {len(tickers)} tickers from {csv_path}')

In [ ]:
records = []
failed  = []

for i, ticker in enumerate(tickers):
    if i % 20 == 0:
        print(f'  [{i}/{len(tickers)}] fetching data...', flush=True)
    raw = fetch_latest_data(ticker, period='120d')
    if raw is None or raw.empty:
        failed.append(ticker)
        continue
    df = calculate_technical_indicators(raw)
    if df.empty:
        failed.append(ticker)
        continue
    last = df.iloc[[-1]].copy()
    last['ticker']   = ticker
    last['exchange'] = _ticker_exchange(ticker)
    records.append(last)

print(f'\nSuccessful: {len(records)}  |  Failed: {len(failed)}')
if failed:
    shown = failed[:15]
    more  = f' ... and {len(failed)-15} more' if len(failed) > 15 else ''
    print(f'Failed: {", ".join(shown)}{more}')

NameError: name 'tickers' is not defined

In [ ]:
if not records:
    raise RuntimeError('No valid ticker data fetched. Check your CSV and network connection.')

snapshot = pd.concat(records, ignore_index=True)

# Cross-sectional percentile rank per exchange (mirrors training)
for feat in FEATURE_NAMES:
    rank_col = f'{feat}_rank'
    snapshot[rank_col] = snapshot.groupby('exchange')[feat].rank(pct=True)
    # Fill NaN for singleton-exchange tickers with global rank
    nan_mask = snapshot[rank_col].isna()
    if nan_mask.any():
        snapshot.loc[nan_mask, rank_col] = snapshot.loc[nan_mask, feat].rank(pct=True)

X = snapshot[ALL_FEATURE_NAMES]
print(f'Feature matrix: {X.shape}  ({X.shape[0]} tickers, {X.shape[1]} features)')

In [ ]:
def top5_by_confidence(model_data, X, ticker_list, label):
    """Return a ranked DataFrame of top-5 tickers by P(UP>3%) confidence."""
    pipeline = model_data['model']
    classes  = list(pipeline.classes_)
    up_idx   = classes.index(1)
    proba    = pipeline.predict_proba(X)[:, up_idx]

    df = pd.DataFrame({'Ticker': ticker_list, 'P(UP>3%)': proba})
    df = df.sort_values('P(UP>3%)', ascending=False).head(5).reset_index(drop=True)
    df.index += 1  # rank 1-5
    df.index.name = 'Rank'
    df['P(UP>3%)'] = df['P(UP>3%)'].map('{:.1%}'.format)
    print(f'\n{"="*40}')
    print(f'  TOP 5  —  {label}')
    print(f'{"="*40}')
    display(df)
    return df

ticker_list = snapshot['ticker'].tolist()

top5_5d  = top5_by_confidence(model_5d,  X, ticker_list, '5-Day Horizon  (P(UP>3%))')
top5_14d = top5_by_confidence(model_14d, X, ticker_list, '14-Day Horizon (P(UP>3%))')